In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 00_data_quality_overview.py
# Purpose of Script: Provide Summary Statistics of Data Quality Checks.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb

In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [4]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [5]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Analysis - Raw SOR DQ ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Results Per File
df = con.execute(f""" select * from '{path_samp}dq_final.parquet'""").df()

In [6]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Raw DQ - Summary Statistics
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Summary - Per Platform
df_summary = (
    df
    .groupby("platform")
    .agg(files=("file", "count"),
        total_rows=("number_of_rows", "sum"),
        earliest_content=("minimum_content_date", "min"),
        latest_content=("maximum_content_date", "max"),
        earliest_sor=("minimum_sor_date", "min"),
        latest_sor=("maximum_sor_date", "max"),
        earliest_application=("minimum_application_date", "min"),
        latest_application=("maximum_application_date", "max"),
        missing_content_dates=("number_content_date_na", "sum"),
        missing_sor_dates=("number_sor_date_na", "sum"),
        missing_application_dates=("number_application_date_na", "sum"),
        territory_failures=("check_terr", lambda x: (x == "Y").sum()),
        uuid_failures=("check_uuids", lambda x: (x == "Y").sum()),
        platform_uuid_failures=("check_plat_uuids", lambda x: (x == "Y").sum()),).reset_index())

df_summary["platform"] = df_summary["platform"].str.capitalize()
df_summary = df_summary.sort_values(by = "total_rows", ascending=False).reset_index(drop=True)

# Add Overview Row
overall = pd.DataFrame({"platform": ["All Platforms"],
                        "files": [len(df)],
                        "total_rows": [df["number_of_rows"].sum()],
                        "earliest_content": [df["minimum_content_date"].min()],
                        "latest_content": [df["maximum_content_date"].max()],
                        "earliest_sor": [df["minimum_sor_date"].min()],
                        "latest_sor": [df["maximum_sor_date"].max()],
                        "earliest_application": [df["minimum_application_date"].min()],
                        "latest_application": [df["maximum_application_date"].max()],
                        "missing_content_dates": [df["number_content_date_na"].sum()],
                        "missing_sor_dates": [df["number_sor_date_na"].sum()],
                        "missing_application_dates": [df["number_application_date_na"].sum()],
                        "territory_failures": [(df["check_terr"] == "Y").sum()],
                        "uuid_failures": [(df["check_uuids"] == "Y").sum()],
                        "platform_uuid_failures": [(df["check_plat_uuids"] == "Y").sum()]})

df_summary = pd.concat([df_summary, overall], ignore_index=True)

In [7]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Outputs
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Overview Table
df_overview = df_summary[['platform','files','total_rows','earliest_content',
                          'latest_content','earliest_sor','latest_sor',
                          'earliest_application','latest_application']]

# Missing and Failures
df_miss_fail = df_summary[['platform','files','total_rows','missing_content_dates',
                           'missing_sor_dates','missing_application_dates',
                           'territory_failures','uuid_failures',
                           'platform_uuid_failures']]

In [14]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_overview.to_csv(f"{path_out}01_dq_1_overview.csv")
df_miss_fail.to_csv(f"{path_out}01_dq_2_miss_fail.csv")

In [15]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Expore Specific Weaknesses
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# UUID Failures
df_uuid_tiktok = df[(df["platform"] == "tiktok") & (df["check_uuids"] == "Y")]
df_uuid_facebook = df[(df['platform'] == 'facebook') & (df['check_uuids'] == "Y")]
df_uuid_instagram = df[(df['platform'] == 'instagram') & (df['check_uuids'] == "Y")]

# Platform UUID Failures
df_plat_uuid_facebook = df[(df['platform'] == 'facebook') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_instagram = df[(df['platform'] == 'instagram') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_tiktok = df[(df['platform'] == 'tiktok') & (df['check_plat_uuids'] == "Y")]
df_plat_uuid_whatsapp = df[(df['platform'] == 'whatsapp') & (df['check_plat_uuids'] == "Y")]

In [21]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Append All Data Quality Failures
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# UUID Failures
df_fail_uuid = pd.concat([df_uuid_tiktok, df_uuid_facebook, df_uuid_instagram]).reset_index()

# Platform UUID Failures
df_fail_plat_uuid = pd.concat([df_plat_uuid_facebook, df_plat_uuid_instagram,
                               df_plat_uuid_tiktok, df_plat_uuid_whatsapp]).reset_index()

In [23]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Data Quality Failures
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_fail_uuid.to_csv(f"{path_out}01_dq_3_uuid_failures.csv")
df_fail_plat_uuid.to_csv(f"{path_out}01_dq_4_plat_uuid_failures.csv")

In [24]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Analysis - Post Cleaning DQ ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_clean = con.execute(f""" select * from '{path_samp}dq_cleaning_final.parquet'""").df()

In [49]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Summarize DQ Clean Results
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Find Total NAs for all records
df_na_summary = (df_clean.groupby("platform", as_index=False).sum(numeric_only=True))
df_na_summary = df_na_summary[df_na_summary.drop(columns="platform").gt(0).any(axis=1)]
df_na_summary["platform"] = df_na_summary["platform"].str.capitalize()

# Join Total SOR Amounts per Platform
df_na_summary = df_na_summary.merge(df_overview[["platform","total_rows"]],
                                    on="platform", how="left")

# Add Overview Row
total_nas = df_na_summary.sum(numeric_only=True).astype(int)
total_nas["platform"] = "All Platforms"
total_nas = total_nas.to_frame().T
df_na_summary = pd.concat([df_na_summary, total_nas], ignore_index=True)

In [52]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_na_summary.to_csv(f"{path_out}01_dq_5_na_summary.csv")

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Expore Specific Weaknesses
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Isolate Data with NAs in Platform Name Column
df_na_fail_plat_name = df_clean[df_clean["p_name"] >= 1]
df_na_fail_plat_name["platform"] = df_na_fail_plat_name["platform"].str.capitalize()
df_na_fail_plat_name = df_na_fail_plat_name.merge(df_overview[["platform","total_rows"]],
                                                  on="platform", how="left")